# 第13章 RAG

## 13.3 RAG 向けに LLM を指示チューニングする

### 13.3.2 指示チューニングしたモデルを LangChain で使う

#### 環境の準備

In [ ]:
!pip install transformers[torch,sentencepiece] langchain langchain-community langchain-huggingface faiss-cpu jq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 381.4 kB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_6

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers.trsiner_utils import set_seed
set_seed(42)

#### Chat Model の作成

In [ ]:
import torch
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
)
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja-aio-retriever"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

tokenizer = AutoTokenzier.from_pretrained(model_name)

generation_config = {
    "max_new_tokens": 32,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
}

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

#### Embedding Model の作成

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = "BAAI/bge-m3"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={"model_kwargs": {"torch_dtype": torch.bfloat16}},
)

#### データストアの構築

In [ ]:
# 検索対象の文書集合のファイルをダウンロード
!wget  \
https://github.com/ghmagazine/llm-book/raw/main/chapter13/docs.json

In [ ]:
from langchain_community.document_loaders import JSONLoader

document_loader = JSONLoader(
    file_path="./docs.json",
    jq_schema=".text",
    json_lines=True,
)

documents = document_loader.load()

In [ ]:
print(len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    add_start_index=True,
)

split_documents = text_splitter.split_documents(documents)

In [ ]:
print(len(split_documents))